# Model A: MobileNetV2 transfer learning
Train a one-logit MobileNetV2 classifier on original 128×128 images. Validation rows are stratified from the persisted training CSV only; the reserved test CSV is used only after loading the best-validation checkpoint.

In [ ]:
from dataclasses import asdict
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader

from applied_ai_midterm.config import load_config
from applied_ai_midterm.data import ImagePathDataset, load_splits
from applied_ai_midterm.evaluation import evaluate_classifier
from applied_ai_midterm.models import build_mobilenet_v2_classifier
from applied_ai_midterm.training import (
    create_train_validation_frames,
    fit_classifier,
    load_best_classifier,
    seed_everything,
    select_device,
)
from applied_ai_midterm.transforms import classifier_transform

In [ ]:
config = load_config(Path("configs/config.yaml"))
seed_everything(config.random_seed)
device = select_device()
splits = load_splits(Path("data/splits"))
train_frame, validation_frame = create_train_validation_frames(
    splits.train, validation_ratio=0.20, random_seed=config.random_seed
)
assert set(train_frame.filepath).isdisjoint(validation_frame.filepath)
assert set(splits.test.filepath).isdisjoint(train_frame.filepath)
assert set(splits.test.filepath).isdisjoint(validation_frame.filepath)
display(pd.DataFrame({"subset": ["train", "validation", "test"], "count": [len(train_frame), len(validation_frame), len(splits.test)]}))
print(f"Device: {device}")

In [ ]:
train_dataset = ImagePathDataset(
    train_frame, classifier_transform(training=True, image_size=config.high_resolution_size)
)
validation_dataset = ImagePathDataset(
    validation_frame, classifier_transform(training=False, image_size=config.high_resolution_size)
)
test_dataset = ImagePathDataset(
    splits.test, classifier_transform(training=False, image_size=config.high_resolution_size)
)
loader_kwargs = {"batch_size": config.classifier_batch_size, "num_workers": config.num_workers}
generator = torch.Generator().manual_seed(config.random_seed)
train_loader = DataLoader(train_dataset, shuffle=True, generator=generator, **loader_kwargs)
validation_loader = DataLoader(validation_dataset, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_dataset, shuffle=False, **loader_kwargs)

In [ ]:
model = build_mobilenet_v2_classifier(freeze_features=True).to(device)
optimizer = torch.optim.Adam(
    (parameter for parameter in model.parameters() if parameter.requires_grad), lr=1e-3
)
checkpoint_path = Path("checkpoints/model_a_best.pt")
history = fit_classifier(
    model,
    train_loader,
    validation_loader,
    optimizer,
    checkpoint_path,
    epochs=config.classifier_epochs,
    device=device,
    config=asdict(config),
)

In [ ]:
history_frame = pd.DataFrame(
    {
        "epoch": [row["epoch"] for row in history],
        "training_loss": [row["training_loss"] for row in history],
        "validation_loss": [row["validation_loss"] for row in history],
    }
)
history_frame.plot(x="epoch", y=["training_loss", "validation_loss"], figsize=(8, 5))
plt.title("Model A loss")
plt.ylabel("BCEWithLogitsLoss")
plt.show()

In [ ]:
best_checkpoint = load_best_classifier(model, checkpoint_path, device)
criterion = nn.BCEWithLogitsLoss()
test_loss, test_metrics, test_labels, test_probabilities = evaluate_classifier(
    model, test_loader, criterion, device
)
display(pd.Series(test_metrics.as_dict(), name="Model A test metrics").to_frame())
print(f"Best validation epoch: {best_checkpoint['epoch']}")
print(f"Test BCE loss: {test_loss:.4f}")